In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from joblib import Parallel, delayed

from gower_similarity.core.similarity import GowerSimilarity

In [ ]:
# load and preprocess data
n_rows = 1000
df = pd.read_csv('<your_path>/adult.csv').head(n_rows)

df = df[['age', 'educational-num', 'race', 'gender', 'hours-per-week', 'relationship', 'occupation', 'education', 'workclass']]

In [ ]:
# process data -> where spot '?' replace with NaN
df.replace(to_replace='?', value=np.nan, inplace=True)

In [ ]:
# define features types
feature_types = {
    "age": "ratio_scale_interval",
    "educational-num": "ratio_scale_interval",
    "race": "categorical_nominal",
    "gender": "categorical_nominal",
    "hours-per-week": "ratio_scale_interval",
    "relationship": "categorical_nominal",
    "occupation": "categorical_nominal",
    "education": "categorical_nominal",
    "workclass": "categorical_nominal"
}

In [ ]:
# fit Gower instance
gs = GowerSimilarity(feature_types=feature_types).fit(df)

In [ ]:
# create helper function to calculate distances for each row
df = df.to_numpy()

def calculate_row(i):
    xi = df[i]
    return [gs.distance(xi, xj) for xj in df]

In [ ]:
# calculate distance matrix in parallel
distance_matrix = np.zeros((len(df), len(df)), dtype=np.float32)
distance_matrix = np.array(Parallel(n_jobs=-1)(delayed(calculate_row)(i) for i in range(len(df)))).astype(np.float32)

In [ ]:
# initialize HDBSCAN
clusterer = HDBSCAN(metric='precomputed', min_cluster_size=10, min_samples=5)
clusterer.fit(distance_matrix)

In [ ]:
# show number of clusters found
print("Cluster labels:", clusterer.labels_.max())

In [ ]:
# evaluate clustering
silhouette_score = silhouette_score(distance_matrix, clusterer.labels_, metric='precomputed')
print("Silhouette Score:", silhouette_score)